In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchsummary import summary
import torchmetrics # For calculating metrics
import time
import sys
from pathlib import Path
import os


In [2]:
project_root = Path(os.getcwd())

# If running from notebooks folder, go one level up
if project_root.name == 'notebooks':
    project_root = project_root.parent

sys.path.append(str(project_root))

In [3]:
from Models.model2 import DeeperCNN

In [4]:
# --- Configuration (Global Constants) ---
DATA_DIR = project_root / 'data' / 'raw' / 'chest_xray'
TRAIN_DIR = DATA_DIR / 'train'
VAL_DIR = DATA_DIR / 'val'
BATCH_SIZE = 32
NUM_EPOCHS = 30
LEARNING_RATE = 1e-3
MODEL_SAVE_PATH = project_root / 'model_2_best.pth'

In [5]:
# --- 1. Define Data Transforms (Global) ---
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # ImageNet std
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

In [6]:
def train_model(model, criterion, optimizer, dataloaders, dataset_sizes, device, num_epochs=25):
    start_time = time.time()
    
    # Trackers for best model
    best_model_weights = model.state_dict()
    best_f1 = 0.0

    # Initialize torchmetrics for metrics
    f1_metric = torchmetrics.F1Score(task="binary").to(device)
    precision_metric = torchmetrics.Precision(task="binary").to(device)
    recall_metric = torchmetrics.Recall(task="binary").to(device)
    accuracy_metric = torchmetrics.Accuracy(task="binary").to(device)

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            running_loss = 0.0
            
            # Reset metrics
            f1_metric.reset()
            precision_metric.reset()
            recall_metric.reset()
            accuracy_metric.reset()

            # Iterate over data
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device).float().view(-1, 1) # Ensure correct shape/type

                # Zero the parameter gradients
                optimizer.zero_grad()

                # Forward pass
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    
                    # Apply sigmoid to outputs for metrics
                    preds = torch.sigmoid(outputs)

                    # Backward + optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # Statistics
                running_loss += loss.item() * inputs.size(0)
                # Update metrics
                f1_metric.update(preds, labels)
                precision_metric.update(preds, labels)
                recall_metric.update(preds, labels)
                accuracy_metric.update(preds, labels)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = accuracy_metric.compute()
            epoch_f1 = f1_metric.compute()
            epoch_precision = precision_metric.compute()
            epoch_recall = recall_metric.compute()

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} F1: {epoch_f1:.4f} Precision: {epoch_precision:.4f} Recall: {epoch_recall:.4f}')

            # Deep copy the model if it's the best one
            if phase == 'val' and epoch_f1 > best_f1:
                best_f1 = epoch_f1
                best_model_weights = model.state_dict()
                torch.save(best_model_weights, str(MODEL_SAVE_PATH))
                print(f'New best model saved to {MODEL_SAVE_PATH} (F1: {best_f1:.4f})')

        print()

    time_elapsed = time.time() - start_time
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val F1: {best_f1:4f}')
    # Load best model weights
    model.load_state_dict(best_model_weights)
    return model

In [7]:
# --- 5. Start Training (Main Execution Block) ---
if __name__ == "__main__":
    print(f"Project root: {project_root}")
    print(f"Train directory: {TRAIN_DIR}")
    print(f"Val directory: {VAL_DIR}")
    
    # Check if data paths exist
    if not TRAIN_DIR.exists() or not VAL_DIR.exists():
        print("="*60)
        print("ERROR: Data directories not found. Please check paths.")
        print(f"       Checked TRAIN_DIR: {TRAIN_DIR.absolute()}")
        print(f"       Checked VAL_DIR: {VAL_DIR.absolute()}")
        print("="*60)
    else:
        print("Data directories found.")
        
        # --- 2. Create DataLoaders ---
        print("Loading data...")
        image_datasets = {
            'train': datasets.ImageFolder(str(TRAIN_DIR), data_transforms['train']),
            'val': datasets.ImageFolder(str(VAL_DIR), data_transforms['val'])
        }

        dataloaders = {
            'train': DataLoader(image_datasets['train'], batch_size=BATCH_SIZE, shuffle=True, num_workers=4),
            'val': DataLoader(image_datasets['val'], batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
        }

        dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
        class_names = image_datasets['train'].classes
        print(f"Class names: {class_names}")
        print(f"Training data size: {dataset_sizes['train']}")
        print(f"Validation data size: {dataset_sizes['val']}")

        # --- 3. Initialize Model, Loss, Optimizer ---
        print("\nInitializing model...")
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {device}")

        model = DeeperCNN().to(device)

        # Print model summary
        print("\nModel Architecture:")
        print("="*60)
        summary(model, input_size=(3, 224, 224))
        print("="*60)

        criterion = nn.BCEWithLogitsLoss()
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
        
        print("Starting training...")
        trained_model = train_model(model, criterion, optimizer, dataloaders, dataset_sizes, device, num_epochs=NUM_EPOCHS)

Project root: d:\Major\FedMedSeg
Train directory: d:\Major\FedMedSeg\data\raw\chest_xray\train
Val directory: d:\Major\FedMedSeg\data\raw\chest_xray\val
Data directories found.
Loading data...
Class names: ['NORMAL', 'PNEUMONIA']
Training data size: 5216
Validation data size: 16

Initializing model...
Using device: cuda

Model Architecture:
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 224, 224]             896
              ReLU-2         [-1, 32, 224, 224]               0
            Conv2d-3         [-1, 32, 224, 224]           9,248
              ReLU-4         [-1, 32, 224, 224]               0
         MaxPool2d-5         [-1, 32, 112, 112]               0
            Conv2d-6         [-1, 64, 112, 112]          18,496
              ReLU-7         [-1, 64, 112, 112]               0
            Conv2d-8         [-1, 64, 112, 112]          36,928
              Re